# Multi-channel ξ PARF — K-EMA × sparse PARF hybrid

This notebook trains the **MultiXiPARFLM** — a hybrid that replaces the
single `causal_cumulative_mean` ξ inside PARFLM with the K-channel K-EMA
multi-resolution ξ that achieved 14.69 PPL as a standalone SPLM.

## Hypothesis

The existing PARF results are handicapped because they use the old single-ξ
design.  With K-EMA underneath:
- Multi-ξ SPLM (no PARF): **14.69 PPL** (best α-init, 4k steps)
- PARF + single ξ (P10i, k=8): **28.00 PPL** (8k steps)

If the ξ-bottleneck was the binding constraint on PARF, this hybrid should
push below 14.69 PPL — the pair-exchange forces would add information that
K-EMA alone cannot capture.

## Arms

| # | Arm | V_φ | top_k | α-init | Description |
|---|-----|-----|------:|--------|-------------|
| 1 | `competitive_k8` | structural_competitive | 8 | [0.25, 0.50, 0.75, 0.95] | Best α-init + best PARF config |
| 2 | `competitive_k4` | structural_competitive | 4 | [0.25, 0.50, 0.75, 0.95] | Lower k for memory savings |
| 3 | `structural_k8`  | structural | 8 | [0.25, 0.50, 0.75, 0.95] | Plain structural V_φ (no softmax) |

All arms use: d=256, L=8, fixed_gamma=0.30, causal_force=True,
learnable α, pilot schedule (4000 steps for quick comparison).

## Hardware

- **A100 40GB**: ~3-4 h per arm (PARF autograd.grad is heavier than pure SPLM)
- Grad-accumulation available via `GRAD_ACCUM` if OOM on smaller GPUs
- TF32 disabled for autograd.grad stability

## 1. Environment setup

In [ ]:
import os, sys, subprocess, shutil, json, time, math
from pathlib import Path

os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')

IN_COLAB = 'google.colab' in sys.modules
print('In Colab:', IN_COLAB)

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/semsimula_parf_multixi')
    REPO_PARENT = Path('/content')
else:
    DRIVE_ROOT = Path.home() / 'semsimula_parf_multixi'
    REPO_PARENT = Path.cwd().parent.parent.parent.parent

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS = DRIVE_ROOT / 'results'
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
print('Drive root   :', DRIVE_ROOT)
print('Results dir  :', DRIVE_RESULTS)

In [ ]:
REPO_URL = 'https://github.com/dimitarpg13/semsimula-paper.git'
REPO_DIR = REPO_PARENT / 'semsimula-paper'

if IN_COLAB:
    if REPO_DIR.exists():
        print(f'Repo already cloned at {REPO_DIR}')
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'],
                       check=False)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL,
                        str(REPO_DIR)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'torch', 'numpy', 'matplotlib', 'tiktoken', 'datasets'],
                   check=True)

SCRIPTS_DIR = REPO_DIR / 'notebooks' / 'conservative_arch' / 'scaleup'
PARF_DIR    = REPO_DIR / 'notebooks' / 'conservative_arch' / 'parf'
MULTIXI_DIR = REPO_DIR / 'notebooks' / 'conservative_arch' / 'multixi'
CA_DIR      = REPO_DIR / 'notebooks' / 'conservative_arch'
assert SCRIPTS_DIR.exists(), f'Missing: {SCRIPTS_DIR}'
assert PARF_DIR.exists(), f'Missing: {PARF_DIR}'
assert MULTIXI_DIR.exists(), f'Missing: {MULTIXI_DIR}'
print('Scripts dir  :', SCRIPTS_DIR)
print('PARF dir     :', PARF_DIR)
print('MultiXi dir  :', MULTIXI_DIR)

## 2. GPU check

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')
    DEVICE = 'cuda'
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    print('TF32 disabled for PARF autograd.grad stability')
elif torch.backends.mps.is_available():
    print('GPU: Apple MPS')
    DEVICE = 'mps'
else:
    print('WARNING: No GPU detected')
    DEVICE = 'cpu'

print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

## 3. Experiment configuration

Edit `ARMS_TO_RUN` to select which arms to train.
The `learned_from_uniform` α-init [0.25, 0.50, 0.75, 0.95] was the
winner from the α-sweep notebook (14.69 PPL).

In [ ]:
# ─── Shared hyperparameters ───
MODE          = 'pilot'      # 4000 steps (quick comparison); change to 'scaleup' for 8000
FIXED_GAMMA   = 0.30
XI_CHANNELS   = 4
XI_ALPHA_INITS = '0.25,0.50,0.75,0.95'   # best from α-sweep
SEED          = 0
MAX_TRAIN_TOK = 5_000_000
GRAD_ACCUM    = 2            # 2 micro-batches to fit PARF on 40GB A100

# ─── Arm definitions ───
ARM_DEFS = {
    'competitive_k8': {
        'v_phi_kind': 'structural_competitive',
        'top_k': 8,
        'desc': 'Competitive V_φ + top-k=8 (best PARF config + best α)',
    },
    'competitive_k4': {
        'v_phi_kind': 'structural_competitive',
        'top_k': 4,
        'desc': 'Competitive V_φ + top-k=4 (lower memory)',
    },
    'structural_k8': {
        'v_phi_kind': 'structural',
        'top_k': 8,
        'desc': 'Plain structural V_φ + top-k=8',
    },
}

# ─── Select arms to run ───
ARMS_TO_RUN = list(ARM_DEFS.keys())

print(f'Will run {len(ARMS_TO_RUN)} arms:')
for name in ARMS_TO_RUN:
    d = ARM_DEFS[name]
    print(f'  {name:25s}  V_φ={d["v_phi_kind"]}  k={d["top_k"]}  ({d["desc"]})')

## 4. Precompute logfreq surprisal (if needed)

In [ ]:
LOGFREQ_PATH = SCRIPTS_DIR / 'results' / 'logfreq_surprisal_tinystories.npy'

if not LOGFREQ_PATH.exists():
    print('Computing logfreq surprisal (one-time, ~2 min)...')
    subprocess.run(
        [sys.executable, str(SCRIPTS_DIR / 'compute_unigram_frequencies_tinystories.py')],
        cwd=str(SCRIPTS_DIR), check=True,
    )
    assert LOGFREQ_PATH.exists()
    print('Done.')
else:
    print(f'logfreq file exists: {LOGFREQ_PATH}')

## 5. Train multi-ξ PARF arms

Each arm trains a `MultiXiPARFLM` with different V_φ / top-k configs.
Completed arms are skipped on re-run.

In [ ]:
TRAINER = str(SCRIPTS_DIR / 'train_parf_multixi_scaleup.py')

results = {}

for arm_name in ARMS_TO_RUN:
    arm_def = ARM_DEFS[arm_name]
    arm_results_dir = DRIVE_RESULTS / arm_name
    arm_results_dir.mkdir(parents=True, exist_ok=True)

    summary_glob = list(arm_results_dir.glob('*_summary.md'))
    if summary_glob:
        print(f'\n── {arm_name}: SKIP (already complete) ──')
        with open(summary_glob[0]) as f:
            for line in f:
                if 'Final' in line:
                    print(f'   {line.strip()}')
        results[arm_name] = {'status': 'skipped'}
        continue

    print(f'\n{"═"*60}')
    print(f'  {arm_name}: {arm_def["desc"]}')
    print(f'{"═"*60}')

    cmd = [
        sys.executable, TRAINER,
        '--mode', MODE,
        '--seed', str(SEED),
        '--fixed-gamma', str(FIXED_GAMMA),
        '--xi-channels', str(XI_CHANNELS),
        '--xi-alpha-inits', XI_ALPHA_INITS,
        '--xi-alpha-init-mode', 'explicit',
        '--max-train-tokens', str(MAX_TRAIN_TOK),
        '--results-dir', str(arm_results_dir),
        '--tag-suffix', arm_name,
        '--logfreq-path', str(LOGFREQ_PATH),
        '--device', DEVICE,
        '--v-phi-kind', arm_def['v_phi_kind'],
        '--top-k', str(arm_def['top_k']),
        '--grad-accum', str(GRAD_ACCUM),
        '--ln-before-distance',
        '--per-layer-v-phi-scale',
        '--grad-checkpoint',
    ]

    t0 = time.time()
    proc = subprocess.run(cmd, cwd=str(SCRIPTS_DIR))
    elapsed = time.time() - t0

    if proc.returncode != 0:
        print(f'  ERROR: trainer exited with code {proc.returncode}')
        results[arm_name] = {'status': 'failed', 'returncode': proc.returncode}
        continue

    summary_files = list(arm_results_dir.glob('*_summary.md'))
    if summary_files:
        with open(summary_files[0]) as f:
            print(f.read())
    results[arm_name] = {
        'status': 'completed',
        'elapsed_min': elapsed / 60,
    }
    print(f'  Completed in {elapsed/60:.1f} min')

print(f'\n{"═"*60}')
print('All arms complete.')
for name, r in results.items():
    print(f'  {name:25s}  {r["status"]}')

## 6. Results comparison

Compare multi-ξ PARF arms against each other and against the baselines:
- Multi-ξ SPLM (no PARF): 14.69 PPL
- PARF + single ξ (P10i, k=8): 28.00 PPL

In [ ]:
import matplotlib.pyplot as plt

# Baselines for context
BASELINES = {
    'Multi-ξ SPLM (no PARF)': 14.69,
    'PARF + single ξ (P10i k=8)': 28.00,
    'Attention baseline': 7.81,
}

arm_ppls = {}
arm_alphas = {}

for arm_name in ARM_DEFS:
    arm_dir = DRIVE_RESULTS / arm_name
    ckpt_files = list(arm_dir.glob('*_ckpt_latest.pt'))
    if not ckpt_files:
        continue
    ckpt = torch.load(ckpt_files[0], map_location='cpu', weights_only=False)
    arm_ppls[arm_name] = ckpt.get('final_val_ppl')
    arm_alphas[arm_name] = ckpt.get('final_xi_alphas')

if not arm_ppls:
    print('No results found yet.')
else:
    print(f'{"Arm":30s} {"Final α":40s} {"Val PPL":>8s}')
    print('─' * 80)
    for name in sorted(arm_ppls, key=lambda x: arm_ppls[x]):
        alpha_str = ', '.join(f'{a:.4f}' for a in arm_alphas[name]) if arm_alphas[name] else '?'
        print(f'{name:30s} [{alpha_str:38s}] {arm_ppls[name]:8.2f}')
    print('─' * 80)
    for bname, bppl in BASELINES.items():
        print(f'{bname:30s} {"":40s} {bppl:8.2f}  (baseline)')

    # Bar chart
    all_names = list(sorted(arm_ppls, key=lambda x: arm_ppls[x])) + list(BASELINES.keys())
    all_ppls  = [arm_ppls[n] for n in sorted(arm_ppls, key=lambda x: arm_ppls[x])] + list(BASELINES.values())
    colors = ['#2ecc71'] * len(arm_ppls) + ['#95a5a6'] * len(BASELINES)

    fig, ax = plt.subplots(figsize=(12, 5))
    bars = ax.barh(all_names, all_ppls, color=colors, edgecolor='white')
    for bar, ppl in zip(bars, all_ppls):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f'{ppl:.2f}', va='center', fontsize=9)
    ax.set_xlabel('Val PPL')
    ax.set_title('Multi-ξ PARF vs baselines (lower is better)')
    ax.invert_yaxis()
    ax.grid(True, axis='x', alpha=0.3)
    fig.tight_layout()
    fig.savefig(DRIVE_RESULTS / 'multixi_parf_comparison.png', dpi=150)
    plt.show()

## 7. Save consolidated report

In [ ]:
report = {
    'experiment': 'parf_multixi_hybrid',
    'config': {
        'mode': MODE,
        'fixed_gamma': FIXED_GAMMA,
        'xi_channels': XI_CHANNELS,
        'xi_alpha_inits': XI_ALPHA_INITS,
        'seed': SEED,
        'grad_accum': GRAD_ACCUM,
    },
    'arms': {},
    'baselines': BASELINES,
}

for name in arm_ppls:
    report['arms'][name] = {
        'v_phi_kind': ARM_DEFS[name]['v_phi_kind'],
        'top_k': ARM_DEFS[name]['top_k'],
        'final_ppl': arm_ppls[name],
        'final_alphas': arm_alphas.get(name),
    }

report_path = DRIVE_RESULTS / 'parf_multixi_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(f'Report saved: {report_path}')
if IN_COLAB:
    print('Results are persisted on Google Drive.')